In [1]:
import os, sys, gc, glob, time, json, re, hashlib, threading, traceback, warnings, random
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

# ponytail: seed management for reproducibility
SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
    
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import cv2
import pydicom
from pydicom.pixel_data_handlers.util import apply_modality_lut
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

# ponytail: TorchIO for medical augmentation (optional)
try:
    import torchio as tio
    HAS_TORCHIO = True
except ImportError:
    HAS_TORCHIO = False

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

T0 = time.time()
KAGGLE_TIME_LIMIT = 8.5 * 3600

def log(msg):
    print(f"[{time.time()-T0:7.1f}s] {msg}", flush=True)

def check_deadline(label=""):
    elapsed = time.time() - T0
    if elapsed > KAGGLE_TIME_LIMIT:
        raise TimeoutError(f"Time budget exceeded ({elapsed/3600:.1f}h) at {label}")

# ── Medical Augmentation (3D-Consistent, No Flips) ──────────────────────────
def augment_3d_consistent(vol, p_rot=0.5, p_scale=0.5, p_intensity=0.5, seed=None):
    """Apply same transform to all slices in volume. No horizontal flips (anatomy-preserving).
    
    vol: (D, H, W) or (D, H, W, C) array
    Returns: augmented volume, same shape
    ponytail: minimal, no TorchIO dependency; use numpy/torch only
    """
    if seed is not None:
        rng = np.random.RandomState(seed)
    else:
        rng = np.random
    
    if isinstance(vol, torch.Tensor):
        vol_np = vol.cpu().numpy() if vol.is_cuda else vol.numpy()
        was_tensor = True
    else:
        vol_np = vol.copy()
        was_tensor = False
    
    # Random affine: ±10° rotation, ±10% scale (consistent across slices)
    if rng.rand() < p_rot:
        angle = rng.uniform(-10, 10) * np.pi / 180
        scale = rng.uniform(0.9, 1.1)
        h, w = vol_np.shape[1:3]
        cx, cy = w / 2, h / 2
        cos_a, sin_a = np.cos(angle), np.sin(angle)
        M = np.array([[cos_a * scale, -sin_a, cx - cx * cos_a * scale + cy * sin_a],
                      [sin_a, cos_a * scale, cy - cx * sin_a - cy * cos_a * scale + cx]])
        for d in range(vol_np.shape[0]):
            vol_np[d] = cv2.warpAffine(vol_np[d], M, (w, h), borderMode=cv2.BORDER_REPLICATE)
    
    # Random intensity: brightness/contrast ±15%, gaussian noise
    if rng.rand() < p_intensity:
        brightness = rng.uniform(0.85, 1.15)
        contrast = rng.uniform(0.85, 1.15)
        vol_np = np.clip(vol_np * contrast * brightness, 0, 255)
        if rng.rand() < 0.5:
            vol_np = vol_np + rng.normal(0, 5, vol_np.shape)
    
    return torch.from_numpy(np.clip(vol_np, 0, 255)).to(torch.uint8) if was_tensor else vol_np

# ── Paths & Path Hardening ───────────────────────────────────────────────────
def _detect_root():
    candidates = [
        Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
        Path("/kaggle/input/rsna-knee-abnormality-detection"),
    ]
    for c in candidates:
        if (c / "test.csv").exists():
            return c
    for d, _, files in os.walk("/kaggle/input"):
        if "test.csv" in files:
            return Path(d)
    return Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")

ROOT        = _detect_root()
WORKING_DIR = Path("/kaggle/working")
ASSET       = Path("/kaggle/input/datasets/tonylica/rsna-knee-bend-dinov3-0917-repro-assets")
DINO_PATH   = Path("/kaggle/input/models/metaresearch/dinov2/pytorch/small/1")

# ── Shared label set ─────────────────────────────────────────────────────────
TARGETS = LAB = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]

# ── CoAtNet arm config ───────────────────────────────────────────────────────
COAT_IMG         = 336
COAT_CROP_MM     = 140.0
COAT_SLOTS       = [("Sagittal", 1, 18), ("Sagittal", 0, 14),
                    ("Coronal",  1, 12), ("Coronal",  0,  8), ("Axial", -1, 12)]
COAT_MAXS        = sum(s[2] for s in COAT_SLOTS)
COAT_K_EVAL      = 62
COAT_NORM        = "imagenet"
COAT_INFER_BATCH = 4

_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

import os

# Full path to your dataset folder
DATASET_PATH = "/kaggle/input/datasets/dreaddevelopment/raptor-knee-maxspan"

COAT_ARMS = [
    {
        "file": "raptor_ft_coatnet_v5_full_swa.pt",
        "path": os.path.join(DATASET_PATH, "raptor_ft_coatnet_v5_full_swa.pt"),
        "arch": "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k",
        "res": 384,
        "w": 0.40,  # Reduced from 1.0 to balance with new arms
    },
    {
        "file": "resnet50_radimagenet.pt",
        "path": "/kaggle/input/datasets/marwanmath/resnet-50-radimagenet-marwan/resnet50_radimagenet.pt",
        "arch": "resnet50",
        "res": 224,
        "w": 0.20,
    },
    {
        "file": "rsna_knee_coat_resgated_ep10.pt",
        "path": "/kaggle/input/datasets/mattiaangeli/rsna-knee-coat-resgated-ep10-top3/rsna_knee_coat_resgated_ep10.pt",
        "arch": "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k",
        "res": 336,
        "w": 0.40,
    },
]

# Verify the file exists before loading
for model in COAT_ARMS:
    if os.path.exists(model["path"]):
        print(f"✅ Ready to load: {model['path']}")
    else:
        print(f"❌ File not found: {model['path']}")
        if os.path.exists(DATASET_PATH):
            print(f"   Contents of directory: {os.listdir(DATASET_PATH)}")

# ── DINOv2 arm config ────────────────────────────────────────────────────────
_CPU_COUNT = os.cpu_count() or 4
DINO_CROP_MM       = 130.0
DINO_CACHE_IMG     = 336
DINO_GROUP         = 3
DINO_N_GROUP_MAX   = 1
DINO_SEED          = 2026
DINO_EVAL_BATCH    = 8
DINO_TIME_BUDGET   = 8.0 * 3600

# ── Additional DINOv2 heads from other datasets ──────────────────────────────
DINO_ADDITIONAL_HEADS = [
    "/kaggle/input/datasets/antoinegg1/rsna-knee-e11-diverse-heads-v20/manifest.json",
    "/kaggle/input/datasets/prvsiyan/rsna-knee-v52-radimagenet-heads/manifest.json",
    "/kaggle/input/datasets/antoinegg1/rsna-knee-e9-radimagenet-heads-v15/manifest.json",
]

# LLM-read labels for soft regularization (optional)
LLM_LABELS_PATH = "/kaggle/input/datasets/pilkwang/rsna-knee-llm-labels/rsna_predictions.csv"
DINO_HDR_THREADS   = min(8, _CPU_COUNT)
DINO_PIX_THREADS   = min(6, _CPU_COUNT)
DINO_ORDER_THREADS = min(8, _CPU_COUNT)

SLOTS_RECOVERED = [
    ("SAG_FLUID_FS",   "Sagittal", True,  True ),
    ("COR_FLUID_FS",   "Coronal",  True,  True ),
    ("AX_FLUID_FS",    "Axial",    True,  True ),
    ("SAG_FLUID_NOFS", "Sagittal", True,  False),
    ("COR_T1",         "Coronal",  False, False),
    ("SAG_T1",         "Sagittal", False, False),
]
DINO_SLOTS  = SLOTS_RECOVERED
DINO_N_SLOT = len(DINO_SLOTS)
DINO_POOL_PARTS = {"cls_mean": 2, "cls_mean_focal": 3}
SLOT_PRIOR_TABLE = {
    "ACL":             (0, 3, 5),
    "MCL":             (1, 4),
    "Medial Meniscus": (0, 1, 3, 4),
    "Lateral Meniscus":(0, 1, 3, 4),
    "Medial OA":       (1, 4, 5),
    "Lateral OA":      (1, 4, 5),
    "PF OA":           (0, 2, 5),
    "Effusion":        (0, 2),
    "Synovitis":       (0, 2),
    "Baker's":         (0,),
    "Contusion":       (0, 1, 2),
    "Fracture":        (0, 1, 2, 4, 5),
}
SLOT_PRIOR_STRENGTH = 0.55
RULES = {"order": "normal", "lat": "centre", "slot_fallback": False, "decode_fill": "nearest"}

DINO_SLICE_BAND  = (0.2, 0.8)
DINO_AUG_ROT_DEG = 8.0
DINO_AUG_SCALE   = 0.08
DINO_AUG_SHIFT   = 0.05
DINO_AUG_INTENSITY = 0.1
LAT_MIN_OFFSET_MM  = 20.0
LEGACY_LAT_OFFSET_MM = 5.0
FATSAT_OPTS = {"FS", "FATSAT", "FAT_SAT", "FSAT"}
_SEP        = re.compile(r"[_\-.]")
_FATSAT_RX  = re.compile(r"\bfs\b|fatsat|fat sat|\bstir\b|\bspair\b|\bspir\b|\bwe\b|water excit|\btirm\b|\bsting\b|\bfatsup\b")
_T1_RX      = re.compile(r"\bt1\b|\bt1w\b")
_T2_RX      = re.compile(r"\bt2\b|\bt2w\b")
_PD_RX      = re.compile(r"\bpd\b|\bpdw\b|proton|\bdp\b|dens")

TTA_OVERLAP     = True
TTA_POOL        = "prob"
TTA_TARGET_POOL = {
    "Fracture":        "max", "Contusion":       "max",
    "Medial Meniscus": "max", "Lateral Meniscus":"max",
    "ACL":             "top2", "MCL":             "top2",
    "Baker's":         "max", "Synovitis":       "original_mean",
}

# ── Final blend weights ───────────────────────────────────────────────────────
COAT_FOCUS_WEIGHT = {
    "ACL": 0.05, "MCL": 0.05, "Medial Meniscus": 0.07,
    "Lateral Meniscus": 0.08, "Contusion": 0.04, "Fracture": 0.08,
}
V18_COATNET_WEIGHT = {label: 0.50 for label in TARGETS}

# ── Device ───────────────────────────────────────────────────────────────────
DEVS   = [torch.device(f"cuda:{i}") for i in range(torch.cuda.device_count())]
DEVICE = DEVS[0] if DEVS else torch.device("cpu")

print(f"device {DEVICE} | gpus {len(DEVS)} | torch {torch.__version__}", flush=True)

# ── CoAtNet backbone + MIL head ──────────────────────────────────────────────
def build_backbone(arch, pretrained=False):
    hybrid = arch.startswith(("maxvit","maxxvit","coatnet","coat_","convnext"))
    is_vit  = (not hybrid) and any(k in arch for k in ("vit","deit","dinov2","eva","beit"))
    kw = dict(pretrained=pretrained, num_classes=0, in_chans=3)
    if is_vit:
        kw.update(global_pool="token", dynamic_img_size=True)
    else:
        kw.update(global_pool="avg")
    return timm.create_model(arch, **kw)

class RaptorClassifier(nn.Module):
    def __init__(self, backbone, F_dim=768, n=12, drop=0.2):
        super().__init__()
        self.backbone = backbone
        self.norm  = nn.LayerNorm(F_dim)
        self.att   = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop), nn.Linear(256, n))
        self.clsW  = nn.Parameter(torch.zeros(n, F_dim))
        self.clsb  = nn.Parameter(torch.zeros(n))
        nn.init.trunc_normal_(self.clsW, std=0.02)
        self.n = n

    def encode(self, x):
        B, K = x.shape[:2]
        f = self.backbone(x.flatten(0, 1))
        return f.view(B, K, -1)

    def head(self, feats):
        h = self.norm(feats)
        a = torch.softmax(self.att(h), dim=1)
        pooled = torch.einsum("bkn,bkf->bnf", a, h)
        return (pooled * self.clsW).sum(-1) + self.clsb

    def head_consensus(self, feats, keep_fraction=0.68):
        h = self.norm(feats)
        score = self.att(h)
        full_att  = torch.softmax(score, dim=1)
        full_pool = torch.einsum("bkn,bkf->bnf", full_att, h)
        full_logits = (full_pool * self.clsW).sum(-1) + self.clsb
        k = max(3, min(int(round(feats.shape[1] * keep_fraction)), feats.shape[1]))
        keep = torch.zeros_like(score, dtype=torch.bool)
        keep.scatter_(1, torch.topk(score, k=k, dim=1, sorted=False).indices, True)
        focused_att  = torch.softmax(score.masked_fill(~keep, -1e4), dim=1)
        focused_pool = torch.einsum("bkn,bkf->bnf", focused_att, h)
        focused_logits = (focused_pool * self.clsW).sum(-1) + self.clsb
        entropy = -(full_att * torch.log(full_att.clamp_min(1e-8))).sum(dim=1)
        entropy = entropy / max(1e-8, float(np.log(max(2, feats.shape[1]))))
        return full_logits, focused_logits, entropy

    def forward(self, x):
        return self.head(self.encode(x))

# ── EfficientNet-B3 + Max-Pooling MIL (Alternative, Grayscale Input) ───────
class EfficientNetMIL(nn.Module):
    """2D EfficientNet-B3 + max-pooling MIL. Grayscale input.
    
    ponytail: simpler than CoAtNet; matches top-team architecture (instruction 3).
    Input: (B, T, 1, H, W) where T = slice count, H=W=256 or 288
    Output: (B, n_classes)
    """
    def __init__(self, n_classes=12, pretrained=True, pool_mode='max'):
        super().__init__()
        # Load EfficientNet-B3 pretrained (will adapt to grayscale)
        self.backbone = timm.create_model('efficientnet_b3', pretrained=pretrained, num_classes=0)
        backbone_dim = self.backbone.num_features
        self.pool_mode = pool_mode
        # Adapt first conv to grayscale (1 channel -> 3 channels: repeat)
        old_conv = self.backbone.conv_stem
        if old_conv.in_channels == 3:
            old_weight = old_conv.weight.data  # (C_out, 3, 3, 3)
            new_conv = nn.Conv2d(1, old_conv.out_channels, kernel_size=3, stride=2, padding=1, bias=False)
            # Average RGB weights to initialize from grayscale
            new_conv.weight.data = old_weight.mean(dim=1, keepdim=True)
            self.backbone.conv_stem = new_conv
        self.head = nn.Linear(backbone_dim, n_classes)
    
    def forward(self, x):
        """x: (B, T, 1, H, W) -> (B, n_classes)"""
        B, T = x.shape[:2]
        # Flatten batch+time
        x = x.reshape(B * T, *x.shape[2:])  # (B*T, 1, H, W)
        # Extract features per slice
        feats = self.backbone(x)  # (B*T, backbone_dim)
        feats = feats.reshape(B, T, -1)  # (B, T, backbone_dim)
        # MIL pooling
        if self.pool_mode == 'max':
            pooled = feats.max(dim=1)[0]  # (B, backbone_dim)
        elif self.pool_mode == 'mean':
            pooled = feats.mean(dim=1)
        else:  # 'attention'
            att = torch.softmax(self.head(feats), dim=1).unsqueeze(-1)
            pooled = (feats * att).sum(dim=1)
        return self.head(pooled)

def load_efficientnet_mil(n_classes=12, pretrained=True, pool_mode='max', device='cuda'):
    """Load EfficientNet-B3 MIL model."""
    model = EfficientNetMIL(n_classes=n_classes, pretrained=pretrained, pool_mode=pool_mode)
    return model.to(device)

def load_coat_model(pt_path, arch_default, res_default, device):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        ck = torch.load(pt_path, map_location="cpu", weights_only=False)
    arch  = ck.get("arch", arch_default)
    ck_res = int(ck.get("res", res_default))
    bb    = build_backbone(arch, pretrained=False)
    model = RaptorClassifier(bb, F_dim=bb.num_features)
    model.load_state_dict(ck["model"], strict=True)
    model.eval().to(device)
    del ck; gc.collect()
    return model, ck_res

# ── DINOv2 arm config ────────────────────────────────────────────────────────
class SlotHead(nn.Module):
    def __init__(self, dim, n_slot, n_out, hidden=256, p=0.2, prior=False):
        super().__init__()
        self.proj     = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU())
        self.slot_emb = nn.Parameter(torch.randn(n_slot, hidden) * 0.02)
        self.query    = nn.Parameter(torch.randn(n_out,  hidden) * 0.02)
        self.drop     = nn.Dropout(p)
        self.out      = nn.Linear(hidden, n_out)
        self.hidden   = hidden
        p_ = torch.zeros(n_out, n_slot)
        if prior and n_slot == len(DINO_SLOTS) and n_out == len(TARGETS):
            for t, slots in SLOT_PRIOR_TABLE.items():
                if t in TARGETS:
                    p_[TARGETS.index(t), list(slots)] = SLOT_PRIOR_STRENGTH
        self.prior = prior
        if prior:
            self.register_buffer("slot_prior", p_)

    def forward(self, x, mask):
        h   = self.proj(x) + self.slot_emb
        att = torch.einsum("bsh,oh->bos", h, self.query) / self.hidden ** 0.5
        if self.prior:
            att = att + self.slot_prior.unsqueeze(0)
        att = att.masked_fill(mask.unsqueeze(1) < 0.5, -10000.0).softmax(-1)
        ctx = self.drop(torch.einsum("bos,bsh->boh", att, h))
        return (ctx * self.out.weight.unsqueeze(0)).sum(-1) + self.out.bias

class DinoModel(nn.Module):
    def __init__(self, backbone, dim, pool="cls_mean", prior=False):
        super().__init__()
        self.backbone = backbone
        self.pool     = pool
        self.head     = SlotHead(dim * DINO_POOL_PARTS[pool], DINO_N_SLOT, len(TARGETS), prior=prior)
        self.register_buffer("mean", torch.tensor([0.485,0.456,0.406]).view(1,3,1,1))
        self.register_buffer("std",  torch.tensor([0.229,0.224,0.225]).view(1,3,1,1))

    def forward(self, imgs, mask, img_size=None):
        B, S = imgs.shape[:2]
        x = imgs.reshape(B*S, *imgs.shape[2:]).float().div_(255.0)
        if img_size is not None and img_size != x.shape[-1]:
            x = F.interpolate(x, size=(img_size,img_size), mode="bilinear", align_corners=False)
        x = (x - self.mean) / self.std
        out   = self.backbone(pixel_values=x).last_hidden_state
        patch = out[:, 1:]
        parts = [out[:, 0], patch.mean(1)]
        if self.pool == "cls_mean_focal":
            k = max(1, patch.shape[1] // 8)
            parts.append(patch.topk(k, dim=1).values.mean(1))
        feat = torch.cat(parts, dim=1).reshape(B, S, -1)
        return self.head(feat, mask)

class WeightsError(RuntimeError):
    pass

FINGERPRINT_TOL = 0.002

def dino_fingerprint(model, dev, img_size, n_slot=None, group=None, seed=None):
    n_slot = DINO_N_SLOT if n_slot is None else n_slot
    group  = DINO_GROUP  if group  is None else group
    seed   = DINO_SEED   if seed   is None else seed
    g      = torch.Generator().manual_seed(seed)
    imgs   = torch.randint(0, 256, (2, n_slot, group, img_size, img_size), generator=g, dtype=torch.uint8).to(dev)
    mask   = torch.ones(2, n_slot, device=dev); mask[1,-1] = 0.0
    model.eval()
    with torch.inference_mode():
        out = model(imgs, mask, img_size).float().cpu().numpy()
    return out

def check_fingerprint(model, dev, img_size, expected, tol=FINGERPRINT_TOL, tag=""):
    got = dino_fingerprint(model, dev, img_size)
    exp = np.asarray(expected, np.float32)
    if got.shape != exp.shape:
        raise WeightsError(f"{tag}fingerprint shape mismatch {got.shape} vs {exp.shape}")
    d = float(np.abs(got - exp).max())
    if d > tol:
        raise WeightsError(f"{tag}fingerprint differs by {d:.4g} (tol {tol})")
    log(f"{tag}fingerprint OK Δ={d:.2g}")

def build_dino_model(unfreeze_last, source=None, variant="small", pool="cls_mean", prior=False):
    from transformers import AutoModel, AutoConfig
    p  = source if source is not None else DINO_PATH
    try:
        bb = AutoModel.from_pretrained(str(p), local_files_only=True)
    except Exception:
        config = AutoConfig.from_pretrained(str(p), local_files_only=True)
        bb = AutoModel.from_config(config)
        
    n_layer = len(bb.encoder.layer)
    for prm in bb.parameters(): prm.requires_grad = False
    for blk in bb.encoder.layer[max(0, n_layer-unfreeze_last):]:
        for prm in blk.parameters(): prm.requires_grad = True
    for prm in bb.layernorm.parameters(): prm.requires_grad = True
    dim = bb.config.hidden_size
    log(f"DINOv2 backbone {n_layer} blocks, last {unfreeze_last} unfrozen, dim {dim}")
    return DinoModel(bb, dim, pool=pool, prior=prior)

# ── Shared DICOM reader factory ───────────────────────────────────────────────
def _make_coat_reader():
    def order_and_meta(sdir):
        fs = glob.glob(sdir + "/*.dcm"); recs=[]; ps_list=[]
        for f in fs:
            try:
                h   = pydicom.dcmread(f, stop_before_pixels=True)
                iop = getattr(h,"ImageOrientationPatient",None)
                ipp = getattr(h,"ImagePositionPatient",None)
                if iop is not None and ipp is not None and len(iop)==6:
                    r=np.array(iop[:3],float); c=np.array(iop[3:],float)
                    n=np.cross(r,c); pos=float(np.dot(np.array(ipp,float),n))
                else:
                    pos=float(getattr(h,"InstanceNumber",0) or 0)
                ps=getattr(h,"PixelSpacing",None); ps=float(ps[0]) if ps else 0.5
                ps_list.append(ps); recs.append((pos,f,ps))
            except Exception:
                recs.append((0.0,f,0.5))
        recs.sort(key=lambda x:x[0])
        med_ps=float(np.median(ps_list)) if ps_list else 0.5
        return [(f,ps) for _,f,ps in recs], med_ps

    def read_px(f):
        d=pydicom.dcmread(f)
        a=apply_modality_lut(d.pixel_array,d).astype(np.float32)
        if str(getattr(d,"PhotometricInterpretation",""))=="MONOCHROME1":
            a=a.max()-a
        return a

    def mm_crop_resize(a, ps):
        h,w=a.shape; cpx=int(round(COAT_CROP_MM/max(ps,1e-3)))
        cpx=min(cpx,min(h,w)); y0=(h-cpx)//2; x0=(w-cpx)//2
        a=a[y0:y0+cpx, x0:x0+cpx]
        return cv2.resize(a,(COAT_IMG,COAT_IMG),interpolation=cv2.INTER_AREA)
    
    return order_and_meta, read_px, mm_crop_resize

def _pick_coat_series(rows, plane, fluid, used):
    cands=[r for r in rows if r["Anatomical_Plane"]==plane and r["SeriesInstanceUID"] not in used]
    if fluid in (0,1):
        pref=[r for r in cands if int(r.get("Fluid_Sensitive",0) or 0)==fluid]
        if pref: return pref[0]
    return cands[0] if cands else None

def build_coat_study(sid, ser_records, tsdir, reader):
    order_and_meta, read_px, mm_crop_resize = reader
    rows=ser_records.get(sid,[])
    vol=np.zeros((COAT_MAXS,COAT_IMG,COAT_IMG),np.uint8)
    idx=0; used=set()
    for plane,fluid,k in COAT_SLOTS:
        r=_pick_coat_series(rows,plane,fluid,used)
        if r is None: idx+=k; continue
        used.add(r["SeriesInstanceUID"])
        files,med_ps=order_and_meta(f"{tsdir}/{sid}/{r['SeriesInstanceUID']}")
        if not files: idx+=k; continue
        n=len(files); lo,hi=int(n*0.02),int(n*0.98)-1; hi=max(hi,lo)
        picks=np.linspace(lo,hi,k).round().astype(int) if n>1 else [0]*k
        arrs=[]; pss=[]
        for p in picks:
            fp,ps=files[min(p,n-1)]
            try: arrs.append(read_px(fp)); pss.append(ps)
            except Exception: arrs.append(None); pss.append(med_ps)
        valid=[a for a in arrs if a is not None]
        if valid:
            allpx=np.concatenate([a.ravel() for a in valid])
            loq,hiq=np.percentile(allpx,[2.0,98.0])
        else: loq,hiq=0.0,1.0
        for a,ps in zip(arrs,pss):
            if idx>=COAT_MAXS: break
            if a is None: idx+=1; continue
            aw=np.clip((a-loq)/(hiq-loq+1e-6),0,1)
            aw=mm_crop_resize(aw,ps if ps>0 else med_ps)
            vol[idx]=(aw*255).astype(np.uint8); idx+=1
        if idx>=COAT_MAXS: break
    mask=(vol.reshape(COAT_MAXS,-1).sum(1)>0).astype(np.uint8)
    return vol,mask

def _eval_centers(mask,D,k):
    valid=np.where(mask>0)[0]
    if len(valid)<3: valid=np.arange(min(3,D))
    lo,hi=int(valid.min()),int(valid.max())
    cs=[c for c in range(lo+1,hi) if c-1>=lo and c+1<=hi]
    if not cs: cs=[max(1,min((lo+hi)//2,D-2))]
    idx=np.linspace(0,len(cs)-1,k).round().astype(int)
    return [cs[i] for i in idx]

def coat_eval_windows(vol,mask,k,res):
    D=vol.shape[0]; cs=_eval_centers(mask,D,k)
    wins=np.empty((len(cs),3,res,res),np.float32)
    for j,c in enumerate(cs):
        c=max(1,min(c,D-2))
        tri=np.stack([vol[c-1],vol[c],vol[c+1]],0).astype(np.float32)/255.0
        t=torch.from_numpy(tri)
        if t.shape[-1]!=res:
            t=F.interpolate(t[None],size=(res,res),mode="bilinear",align_corners=False)[0]
        wins[j]=t.numpy()
    x=torch.from_numpy(wins)
    x=(x-_MEAN)/_STD
    return x

# ── DINOv2 DICOM helpers ──────────────────────────────────────────────────────
HDR_TAGS = ["SeriesDescription","SequenceName","ScanOptions","ScanningSequence",
            "RepetitionTime","EchoTime","Laterality","PixelSpacing","Rows","Columns",
            "RescaleSlope","RescaleIntercept","ImagePositionPatient","ImageOrientationPatient"]
DECODE_FAILED = []
_NKY = re.compile(r"(\d+)")

def _hdr_vec(s,n):
    if not isinstance(s,str): return None
    try: v=[float(x) for x in s.split("|")]
    except ValueError: return None
    return np.array(v) if len(v)>=n else None

def side_from_geometry(h):
    cx={}
    for r in h.itertuples(index=False):
        ipp=_hdr_vec(getattr(r,"ImagePositionPatient",None),3)
        iop=_hdr_vec(getattr(r,"ImageOrientationPatient",None),6)
        ps =_hdr_vec(getattr(r,"PixelSpacing",None),2)
        rows,cols=(getattr(r,"Rows",None),getattr(r,"Columns",None))
        if ipp is None or iop is None or ps is None or not rows or not cols: continue
        try:
            c=ipp[:3]+iop[:3]*ps[1]*float(cols)/2+iop[3:6]*ps[0]*float(rows)/2
        except (TypeError,ValueError): continue
        cx.setdefault(r.StudyInstanceUID,[]).append(float(c[0]))
    out={}
    for st,xs in cx.items():
        m=float(np.median(xs))
        out[st]=None if abs(m)<LAT_MIN_OFFSET_MM else "R" if m<0 else "L"
    return out

def side_from_corner_x(h):
    out={}
    for st,g in h.groupby("StudyInstanceUID"):
        xs=[]
        for r in g.itertuples(index=False):
            ipp=_hdr_vec(getattr(r,"ImagePositionPatient",None),3)
            if ipp is not None and np.isfinite(ipp).all(): xs.append(float(ipp[0]))
        if not xs: out[st]=None; continue
        x=float(np.median(xs))
        out[st]=None if abs(x)<LEGACY_LAT_OFFSET_MM else "R" if x<0 else "L"
    return out

def lat_of(h, tag=""):
    geo=side_from_corner_x(h) if RULES["lat"]=="corner_x" else side_from_geometry(h)
    d,n_tag,n_geo,n_none,n_dis=({},0,0,0,0)
    for st,g in h.groupby("StudyInstanceUID"):
        v=[str(x).strip().upper() for x in g["Laterality"].dropna()]
        v=[x[0] for x in v if x and x[0] in ("L","R")]
        side=v[0] if v else None
        if side is not None:
            n_tag+=1
            if geo.get(st) is not None and geo[st]!=side: n_dis+=1
        else:
            side=geo.get(st); n_geo+=side is not None; n_none+=side is None
        d[st]=side
    log(f"{tag}lat: {n_tag} from tag, {n_geo} geometry, {n_none} unresolved, {n_dis} disagree")
    return d

def dino_probe(item):
    split,study,series,path=item
    row={"split":split,"StudyInstanceUID":study,"SeriesInstanceUID":series,"dir":path}
    try:
        files=sorted(e.name for e in os.scandir(path) if e.name.endswith(".dcm"))
        row["files"]=files; row["n_slices"]=len(files)
        if not files: return row
        ds=pydicom.dcmread(os.path.join(path,files[len(files)//2]), stop_before_pixels=True,force=True)
        for t in HDR_TAGS:
            v=getattr(ds,t,None)
            if v is None: row[t]=None
            elif isinstance(v,(list,tuple)) or type(v).__name__=="MultiValue":
                row[t]="|".join(str(x) for x in v)
            else: row[t]=str(v)
    except Exception as exc: row["err"]=str(exc)[:120]
    return row

def dino_walk(split):
    base=ROOT/split; items=[]
    if not base.is_dir():
        return pd.DataFrame(columns=["split","StudyInstanceUID","SeriesInstanceUID","dir","files","n_slices"]+HDR_TAGS)
    for study in os.scandir(base):
        if study.is_dir():
            for series in os.scandir(study.path):
                if series.is_dir():
                    items.append((split,study.name,series.name,series.path))
    with ThreadPoolExecutor(max_workers=DINO_HDR_THREADS) as pool:
        rows=list(pool.map(dino_probe,items))
    return pd.DataFrame(rows)

def dino_annotate(df):
    desc=(df["SeriesDescription"].fillna("")+" "+df["SequenceName"].fillna("")).str.lower().str.replace(_SEP," ",regex=True)
    opts=df["ScanOptions"].fillna("").str.upper().str.split("|")
    opts_fs=opts.apply(lambda ts: any(t.strip() in FATSAT_OPTS for t in ts))
    df["fatsat"]=desc.str.contains(_FATSAT_RX)|opts_fs
    tr=pd.to_numeric(df["RepetitionTime"],errors="coerce")
    te=pd.to_numeric(df["EchoTime"],errors="coerce")
    gre=df["ScanningSequence"].fillna("").str.upper().str.contains("GR")
    t1,t2,pdw=(desc.str.contains(_T1_RX),desc.str.contains(_T2_RX),desc.str.contains(_PD_RX))
    df["weight"]=np.where(t1&~t2&~pdw,"T1",np.where(t2&~pdw,"T2",np.where(pdw,"PD",
        np.where(gre,"GRE",np.where(tr<800,"T1",np.where(te>60,"T2",np.where(tr>=800,"PD","UNK")))))))
    df["fluid"]=np.isin(df["weight"],["PD","T2"])
    df["px"]=pd.to_numeric(df["PixelSpacing"].fillna("").str.split("|").str[0].replace("",np.nan),errors="coerce")
    return df

def dino_pick_slots(series_df, plane_map):
    series_df=series_df.copy()
    series_df["plane"]=series_df["SeriesInstanceUID"].map(plane_map)
    out={}
    for study,g in series_df.groupby("StudyInstanceUID"):
        chosen={}
        for name,plane,fluid,fs in DINO_SLOTS:
            sel=(g["plane"]==plane)&(g["fatsat"]==fs)
            if fluid is not None: sel&=g["fluid"]==fluid
            cand=g[sel]
            if len(cand)==0 and RULES["slot_fallback"] and (fluid is False):
                cand=g[(g["plane"]==plane)&~g["fatsat"]]
            if len(cand):
                chosen[name]=cand.sort_values("n_slices",ascending=False).iloc[0]
        out[study]=chosen
    return out

def _natural_key(name):
    return tuple(int(x) if x.isdigit() else x.lower() for x in _NKY.split(str(name)))

def dino_order_slices(rec):
    files,d=(rec.get("ordered") or rec["files"],rec["dir"])
    keyed=[]
    for f in files:
        k=None
        try:
            ds=pydicom.dcmread(os.path.join(d,f),force=True,stop_before_pixels=True,
                               specific_tags=[(32,50),(32,55),(32,19)])
            iop=np.asarray(ds.ImageOrientationPatient,dtype=float)
            ipp=np.asarray(ds.ImagePositionPatient,dtype=float)
            k=float(np.dot(ipp,np.cross(iop[:3],iop[3:])))
        except Exception:
            try: k=float(ds.InstanceNumber)
            except Exception: k=None
        keyed.append((k,f))
    if any(k is None for k,_ in keyed): return (files,False)
    return ([f for _,f in sorted(keyed,key=lambda t:t[0])],True)

def _dino_order_worker(job):
    _, _, _, rec = job
    return dino_order_slices(rec)

def dino_read_slot(rec, n_slice=None, out_size=None):
    n_slice=DINO_GROUP if n_slice is None else n_slice
    out_size=DINO_CACHE_IMG if out_size is None else out_size
    files,d,px=(rec.get("ordered") or rec["files"],rec["dir"],rec["px"])
    n=len(files)
    if n==0: return None
    lo,hi=(int(DINO_SLICE_BAND[0]*(n-1)),int(DINO_SLICE_BAND[1]*(n-1)))
    idx=np.unique(np.linspace(lo,hi,n_slice).astype(int)) if hi>lo else np.array([n//2])
    while len(idx)<n_slice: idx=np.append(idx,idx[-1])
    planes=[]
    for i in idx[:n_slice]:
        try:
            ds=pydicom.dcmread(os.path.join(d,files[int(i)]),force=True)
            a=ds.pixel_array.astype(np.float32)
            sl=float(getattr(ds,"RescaleSlope",1) or 1)
            ic=float(getattr(ds,"RescaleIntercept",0) or 0)
            a=a*sl+ic
        except Exception: a=None
        planes.append(a)
    got=[k for k,p in enumerate(planes) if p is not None]
    if not got:
        DECODE_FAILED.append(rec.get("SeriesInstanceUID",d)); return None
    if len(got)<len(planes):
        DECODE_FAILED.append(rec.get("SeriesInstanceUID",d))
        for k,p in enumerate(planes):
            if p is None: planes[k]=planes[min(got,key=lambda j:abs(j-k))]
    shp=planes[0].shape
    planes=[p if p.shape==shp else np.zeros(shp,np.float32) for p in planes]
    vol=np.stack(planes)
    if px and np.isfinite(px) and px>0:
        want=int(round(DINO_CROP_MM/px)); h,w=shp
        if 16<want<min(h,w):
            cy,cx=h//2,w//2; half=want//2
            vol=vol[:,max(0,cy-half):cy+half,max(0,cx-half):cx+half]
    lo_v,hi_v=np.percentile(vol,[1,99])
    vol=np.clip((vol-lo_v)/max(hi_v-lo_v,1e-6),0,1)
    t=torch.from_numpy(np.ascontiguousarray(vol)).unsqueeze(0)
    t=F.interpolate(t,size=(out_size,out_size),mode="bilinear",align_corners=False)
    return (t.squeeze(0)*255).round().clamp(0,255).to(torch.uint8)

def _dino_decode_worker(job):
    _, _, _, rec = job
    return dino_read_slot(rec, DINO_GROUP*DINO_N_GROUP_MAX, DINO_CACHE_IMG)

def normalise_laterality(img, plane, lat):
    if lat!="R": return img
    if plane in ("Coronal","Axial"): return torch.flip(img,dims=[-1])
    return torch.flip(img,dims=[0])

# ── CoAtNet Inference ────────────────────────────────────────────────────────
def rankpct(x):
    arr=np.asarray(x)
    order=arr.argsort(axis=0).argsort(axis=0).astype(np.float64)
    return order/max(1,arr.shape[0]-1)

@torch.inference_mode()
def coat_infer_probs_batch(model, xwins_list, device):
    """Batch multiple study windows together for GPU efficiency."""
    x = torch.stack(xwins_list).to(device)
    use_cuda = (device != torch.device("cpu") and str(device).startswith("cuda"))
    def _fwd():
        feats = model.encode(x)
        fl, fc, ent = model.head_consensus(feats, keep_fraction=0.68)
        return (torch.sigmoid(fl.float()).cpu().numpy(),
                torch.sigmoid(fc.float()).cpu().numpy(),
                ent.float().cpu().numpy())
    if use_cuda:
        try:
            with torch.autocast("cuda", dtype=torch.float16): return _fwd()
        except RuntimeError as e:
            torch.cuda.empty_cache()
            log(f"[CoAtNet] fp16->fp32 retry: {type(e).__name__}")
            return _fwd()
    return _fwd()

def _coat_worker(arm_idx, arm, study_chunk, chunk_indices, SER, tsdir, reader, device):
    """Run CoAtNet inference for a chunk of studies on a specific GPU."""
    model, res = load_coat_model(find_weight_file(arm["file"]), arm["arch"], arm["res"], device)
    results = {}
    batch_xwins = []
    batch_indices = []

    for local_i, (global_i, sid) in enumerate(zip(chunk_indices, study_chunk)):
        check_deadline(f"coat_worker_{arm_idx}_study_{global_i}")
        try:
            vol, mask = build_coat_study(sid, SER, tsdir, reader)
            xw = coat_eval_windows(vol, mask, k=COAT_K_EVAL, res=res)
            batch_xwins.append(xw)
            batch_indices.append(global_i)
        except Exception as e:
            log(f"  [arm {arm_idx}] study {global_i} FALLBACK ({type(e).__name__})")
            results[global_i] = (
                np.full(len(LAB), 0.5, dtype=np.float32),
                np.full(len(LAB), 0.5, dtype=np.float32),
                np.full(len(LAB), 1.0, dtype=np.float32)
            )

        if len(batch_xwins) == COAT_INFER_BATCH or local_i == len(study_chunk) - 1:
            if batch_xwins:
                try:
                    fps, fcs, ents = coat_infer_probs_batch(model, batch_xwins, device)
                    for bi, g_idx in enumerate(batch_indices):
                        results[g_idx] = (fps[bi], fcs[bi], ents[bi])
                except Exception as e:
                    log(f"  [arm {arm_idx}] batch starting at {batch_indices[0]} FAILED ({type(e).__name__})")
                    for g_idx in batch_indices:
                        results[g_idx] = (
                            np.full(len(LAB), 0.5, dtype=np.float32),
                            np.full(len(LAB), 0.5, dtype=np.float32),
                            np.full(len(LAB), 1.0, dtype=np.float32)
                        )
                batch_xwins = []
                batch_indices = []

    del model; gc.collect()
    if device.type == "cuda":
        with torch.cuda.device(device): torch.cuda.empty_cache()
    return results

def find_weight_file(fname):
    direct=[f"/kaggle/input/raptor-knee-arms/{fname}",
            f"/kaggle/input/raptor-knee-arms/1/{fname}",
            f"/kaggle/input/raptor-cnn336/{fname}"]
    for p in direct:
        if os.path.exists(p): return p
    for d in sorted(glob.glob("/kaggle/input/*/")):
        if "competition" in d.lower(): continue
        hits=glob.glob(os.path.join(d,"**",fname),recursive=True)
        if hits: return hits[0]
    raise RuntimeError(f"{fname} not found under /kaggle/input")

def run_coatnet():
    t0=time.time()
    tsdir=str(ROOT/"test_series")
    if not os.path.isdir(tsdir): tsdir=str(ROOT/"test_images")
    log(f"test root: {ROOT} | series dir: {tsdir}")
    
    test=pd.read_csv(ROOT/"test.csv"); test["StudyInstanceUID"]=test["StudyInstanceUID"].astype(str)
    test_ids=test["StudyInstanceUID"].tolist()
    tser=pd.read_csv(ROOT/"test_series.csv")
    tser["StudyInstanceUID"]=tser["StudyInstanceUID"].astype(str)
    tser["SeriesInstanceUID"]=tser["SeriesInstanceUID"].astype(str)
    SER={k:v.to_dict("records") for k,v in tser.groupby("StudyInstanceUID")}
    log(f"test studies {len(test_ids)} | test series {len(tser)}")
    
    reader=_make_coat_reader()
    N = len(test_ids)
    A = len(COAT_ARMS)
    n_devs = len(DEVS) if DEVS else 1
    
    arm_probs   = [np.full((N, len(LAB)), 0.5, np.float32) for _ in range(A)]
    focus_probs = [np.full((N, len(LAB)), 0.5, np.float32) for _ in range(A)]
    attn_ent    = [np.full((N, len(LAB)), 1.0, np.float32) for _ in range(A)]
    
    for a, arm in enumerate(COAT_ARMS):
        chunk_size = (N + n_devs - 1) // n_devs
        chunks = []
        for i in range(n_devs):
            start = i * chunk_size
            end = min((i + 1) * chunk_size, N)
            if start < end:
                chunks.append((test_ids[start:end], list(range(start, end))))
                
        futures = []
        with ThreadPoolExecutor(max_workers=n_devs) as executor:
            for i, (chunk, indices) in enumerate(chunks):
                dev = DEVS[i % len(DEVS)] if DEVS else torch.device("cpu")
                futures.append(executor.submit(_coat_worker, a, arm, chunk, indices, SER, tsdir, reader, dev))
                
        for f in futures:
            res_map = f.result()
            for g_i, (fp, fc, ent) in res_map.items():
                arm_probs[a][g_i] = fp
                focus_probs[a][g_i] = fc
                attn_ent[a][g_i]  = ent

    _w=np.array([float(arm.get("w",1.0)) for arm in COAT_ARMS],np.float64)
    _w=_w/_w.sum()
    full_rank=np.tensordot(_w, np.stack([rankpct(np.clip(p,0,1)) for p in arm_probs]),axes=(0,0))
    focus_rank=np.tensordot(_w, np.stack([rankpct(np.clip(p,0,1)) for p in focus_probs]),axes=(0,0))
    ranks=full_rank.copy()
    
    for ti,tgt in enumerate(LAB):
        w=float(COAT_FOCUS_WEIGHT.get(tgt,0.0))
        if w>0:
            ranks[:,ti]=(1-w)*full_rank[:,ti]+w*focus_rank[:,ti]
    ranks=np.stack([rankpct(ranks[:,ti:ti+1])[:,0] for ti in range(len(LAB))],axis=1)
    if not np.isfinite(ranks).all(): ranks[~np.isfinite(ranks)]=0.5
    
    sub=pd.DataFrame(ranks.astype(np.float32),columns=LAB)
    sub.insert(0,"StudyInstanceUID",test_ids)
    out=str(WORKING_DIR/"submission_coatnet.csv")
    sub.to_csv(out,index=False)
    log(f"CoAtNet done → {out} | {len(sub)} rows")
    return sub

try:
    coat_sub=run_coatnet()
except Exception as _e:
    log(f"CoAtNet branch failed: {type(_e).__name__}: {_e}")
    traceback.print_exc()
    coat_sub=None

# ── DINOv2 cache builder ─────────────────────────────────────────────────────
def dino_build_cache(slot_map, plane_map, lat_map, tag):
    studies=sorted(slot_map)
    sidx={s:i for i,s in enumerate(studies)}
    CACHE_SLICES=DINO_GROUP*DINO_N_GROUP_MAX
    cache=np.zeros((len(studies),DINO_N_SLOT,CACHE_SLICES,DINO_CACHE_IMG,DINO_CACHE_IMG),np.uint8)
    mask=np.zeros((len(studies),DINO_N_SLOT),np.float32)
    log(f"{tag}: cache {cache.shape} = {cache.nbytes/1024**3:.1f} GB")
    jobs=[(st,k,plane,slot_map[st][name])
          for st in studies
          for k,(name,plane,_,_) in enumerate(DINO_SLOTS)
          if name in slot_map[st]]
    log(f"{tag}: ordering {len(jobs)} slot-series")
    
    with ThreadPoolExecutor(max_workers=DINO_ORDER_THREADS) as pool:
        for (st,k,plane,rec),(files,good) in zip(jobs,pool.map(_dino_order_worker,jobs)):
            rec["ordered"]=files
    log(f"{tag}: decoding {len(jobs)} slot-series")
    
    with ThreadPoolExecutor(max_workers=DINO_PIX_THREADS) as pool:
        for (st,k,plane,_),img in zip(jobs,pool.map(_dino_decode_worker,jobs)):
            if img is None: continue
            cache[sidx[st],k]=normalise_laterality(img,plane,lat_map.get(st)).numpy()
            mask[sidx[st],k]=1.0
    log(f"{tag}: {int(mask.sum())}/{len(jobs)} slots filled")
    gc.collect()
    return studies,cache,mask

# ── TTA window helpers ────────────────────────────────────────────────────────
def dino_window_starts(n_slice,group,overlap=None):
    overlap=TTA_OVERLAP if overlap is None else overlap
    if overlap and n_slice>=group: return list(range(n_slice-group+1))
    return [g*group for g in range(max(n_slice//group,1))]

def dino_augment(imgs,generator=None):
    lead=imgs.shape[:-3]
    x=imgs.reshape(-1,*imgs.shape[-3:]).float()
    n,dev=x.shape[0],x.device
    rot=(torch.rand(n,device=dev,generator=generator)-0.5)*2*(DINO_AUG_ROT_DEG*np.pi/180)
    sc=1.0+torch.rand(n,device=dev,generator=generator)*DINO_AUG_SCALE
    tx=(torch.rand(n,device=dev,generator=generator)-0.5)*2*DINO_AUG_SHIFT
    ty=(torch.rand(n,device=dev,generator=generator)-0.5)*2*DINO_AUG_SHIFT
    cos,sin=torch.cos(rot)/sc,torch.sin(rot)/sc
    theta=torch.zeros(n,2,3,device=dev,dtype=torch.float32)
    theta[:,0,0],theta[:,0,1],theta[:,0,2]=cos,-sin,tx
    theta[:,1,0],theta[:,1,1],theta[:,1,2]=sin,cos,ty
    grid=F.affine_grid(theta,x.shape,align_corners=False)
    x=F.grid_sample(x,grid,mode="bilinear",padding_mode="border",align_corners=False)
    scale=1.0+(torch.rand(n,1,1,1,device=dev,generator=generator)-0.5)*2*DINO_AUG_INTENSITY
    x=(x*scale).clamp(0,255)
    return x.reshape(*lead,*x.shape[-3:]).to(imgs.dtype)

@torch.inference_mode()
def dino_predict_member(model,cache,mask,idx,dev,img_size,group=None,starts=None,jitter=False,jitter_seed=DINO_SEED):
    group=DINO_GROUP if group is None else group
    starts=dino_window_starts(cache.shape[2],group) if starts is None else list(starts)
    target_idx={t:j for j,t in enumerate(TARGETS)}
    jitter_gen=torch.Generator(device=dev); jitter_gen.manual_seed(int(jitter_seed)%(2**63-1))
    model.eval(); out=[]
    
    for b in range(0,len(idx),DINO_EVAL_BATCH):
        sel=idx[b:b+DINO_EVAL_BATCH]
        m=torch.from_numpy(mask[sel]).to(dev)
        win_probs,win_logits,win_orig=([],[],[])
        for st in starts:
            rows=torch.from_numpy(np.ascontiguousarray(cache[sel,:,st:st+group])).to(dev)
            views=[rows]+([dino_augment(rows,generator=jitter_gen)] if jitter else [])
            vp,vl=([],[])
            for view in views:
                with torch.autocast("cuda",enabled=dev.type=="cuda"):
                    z=model(view,m,img_size).float()
                vl.append(z); vp.append(torch.sigmoid(z))
            win_logits.append(torch.stack(vl).mean(0))
            win_probs.append(torch.stack(vp).mean(0))
            win_orig.append(vp[0])
        
        probs=torch.stack(win_probs); logits=torch.stack(win_logits); orig=torch.stack(win_orig)
        v=torch.sigmoid(logits.mean(0)) if TTA_POOL=="logit" else probs.mean(0)
        
        for tgt,mode in TTA_TARGET_POOL.items():
            j=target_idx[tgt]
            if   mode=="max":         v[:,j]=probs[:,:,j].max(0).values
            elif mode=="mean":        v[:,j]=probs[:,:,j].mean(0)
            elif mode=="original_mean": v[:,j]=orig[:,:,j].mean(0)
            elif mode in ("top2","top3"):
                k=min(int(mode[3:]),probs.shape[0]); v[:,j]=probs[:,:,j].topk(k,dim=0).values.mean(0)
        out.append(v.cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0,len(TARGETS)),np.float32)

BUILD_LOCK=threading.Lock(); STATE_LOCK=threading.Lock()

def _dino_run_member(path,m,dev,Cte,Mte,idx,starts):
    with BUILD_LOCK:
        if "state" in m:
            state,fp=(m["state"],None)
        else:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                ck=torch.load(Path(path)/m["file"],map_location="cpu",weights_only=False)
            state,fp=(ck["model"],ck.get("fingerprint"))
        model=build_dino_model(
            int(m["config"]["unfreeze_last"]),
            variant=m["config"]["variant"],
            pool=m["config"].get("pool","cls_mean"),
            prior=bool(m["config"].get("prior",False))
        ).to(dev)
        model.load_state_dict(state)
        if fp is not None:
            check_fingerprint(model,dev,DINO_CACHE_IMG,fp,tag=f"{m['id']}: ")
        else:
            log(f"  {m['id']}: no fingerprint — accepted at reduced weight")
    
    jseed=DINO_SEED+int(hashlib.sha256(str(m["id"]).encode()).hexdigest()[:8],16)
    p=dino_predict_member(model,Cte,Mte,idx,dev,DINO_CACHE_IMG,starts=starts,jitter=True,jitter_seed=jseed)
    del model,state; gc.collect()
    if dev.type=="cuda":
        with torch.cuda.device(dev): torch.cuda.empty_cache()
    return p

def _dino_combine(per_member):
    all_ids=sorted({s for m in per_member for s in m["ids"]})
    pos={s:i for i,s in enumerate(all_ids)}
    acc=np.zeros((len(all_ids),len(TARGETS)),np.float64)
    tot=np.zeros(len(TARGETS),np.float64)
    for m in per_member:
        tw=m.get("target_weight")
        w=np.asarray(tw if tw is not None else [float(m.get("weight",1.0))]*len(TARGETS),np.float64)
        r=pd.DataFrame(m["pred"]).rank(pct=True).to_numpy()
        acc[[pos[s] for s in m["ids"]]]+=r*w[None,:]
        tot+=w
    return all_ids, acc/tot[None,:]

def dino_write_submission(pred,studies,test_df,path):
    sub=pd.DataFrame(pd.DataFrame(pred).rank(pct=True).values,columns=TARGETS)
    sub.insert(0,"StudyInstanceUID",studies)
    sub=test_df[["StudyInstanceUID"]].merge(sub,on="StudyInstanceUID",how="left")
    sub[TARGETS]=sub[TARGETS].fillna(0.5)
    sub.to_csv(path,index=False)
    return sub

def run_dinov2():
    path=ASSET/"rsna-knee-weights"
    man=json.loads((path/"manifest.json").read_text())
    members=man["members"]
    
    # ── Load additional DINOv2 heads ──────────────────────────────────────
    for extra_path in DINO_ADDITIONAL_HEADS:
        try:
            extra_file = Path(extra_path)
            if extra_file.exists():
                extra = json.loads(extra_file.read_text())
                extra_members = extra.get("members", [])
                # Reduce weight for additional heads to avoid over-fitting
                for m in extra_members:
                    m["weight"] = m.get("weight", 1.0) * 0.7
                    m["source"] = extra_path  # Track source
                members.extend(extra_members)
                log(f"Loaded {len(extra_members)} additional DINO heads from {extra_path.name}")
            else:
                log(f"⚠️  Additional DINO path not found: {extra_path}")
        except Exception as e:
            log(f"⚠️  Failed to load extra DINO heads: {e}")
    
    log(f"DINOv2 package: {len(members)} member(s) total (including {len(members)-man['members']} additional)")
    
    test_df=pd.read_csv(ROOT/"test.csv")
    test_series=pd.read_csv(ROOT/"test_series.csv")
    plane_map=dict(zip(test_series["SeriesInstanceUID"],test_series["Anatomical_Plane"]))
    hte=dino_annotate(dino_walk("test_series"))
    log(f"test header pass: {len(hte)} series")
    slot_map=dino_pick_slots(hte,plane_map)
    lat_map=lat_of(hte,"test ")
    
    st_te,Cte,Mte=dino_build_cache(slot_map,plane_map,lat_map,"test")
    idx=np.arange(len(st_te))
    starts=dino_window_starts(Cte.shape[2],DINO_GROUP)
    per_member=[]
    
    for i, m in enumerate(members):
        check_deadline(f"dinov2_member_{m['id']}")
        try:
            dev = DEVS[i % len(DEVS)] if DEVS else torch.device("cpu")
            p=_dino_run_member(str(path),m,dev,Cte,Mte,idx,starts)
            if float(np.std(p))<1e-9:
                log(f"  {m['id']}: degenerate — skipped"); continue
            per_member.append({"id":m["id"],"fold":m.get("fold"),
                               "ids":st_te,"pred":p,"weight":m.get("weight",1.0)})
            all_ids,acc=_dino_combine(per_member)
            dino_write_submission(acc,all_ids,test_df,str(WORKING_DIR/"submission.csv"))
            log(f"  banked {m['id']} ({len(per_member)} total)")
        except Exception as e:
            log(f"  MEMBER {m['id']} FAILED: {type(e).__name__}: {e}")
            
    del Cte,Mte; gc.collect()
    torch.cuda.empty_cache()
    if not per_member: raise WeightsError("no DINOv2 member produced predictions")
    all_ids,acc=_dino_combine(per_member)
    sub=dino_write_submission(acc,all_ids,test_df,str(WORKING_DIR/"submission.csv"))
    log(f"DINOv2 done → submission.csv | {sub.shape}")
    return sub

try:
    dino_sub=run_dinov2()
except Exception as _e:
    log(f"DINOv2 branch failed: {type(_e).__name__}: {_e}")
    traceback.print_exc()
    dino_sub=None

# ── Final Blend — V18 CoAtNet + DINOv2 Rank-Mean ──────────────────────────────
import shutil as _shutil

_primary = WORKING_DIR/"submission.csv"
_backup  = WORKING_DIR/".submission_transformer_backup.csv"

if _primary.is_file():
    _shutil.copy2(_primary, _backup)

try:
    if coat_sub is None:
        raise RuntimeError("CoAtNet output not available")
    if dino_sub is None:
        raise RuntimeError("DINOv2 output not available")
    
    coat_path = WORKING_DIR/"submission_coatnet.csv"
    dino_path = _primary
    coat_df = pd.read_csv(coat_path, dtype={"StudyInstanceUID":str})
    dino_df = pd.read_csv(dino_path, dtype={"StudyInstanceUID":str})
    blend_labels = [c for c in dino_df.columns if c!="StudyInstanceUID"]
    
    if coat_df.columns.tolist()!=dino_df.columns.tolist():
        raise RuntimeError("schema mismatch between CoAtNet and DINOv2 submissions")
    if coat_df["StudyInstanceUID"].tolist()!=dino_df["StudyInstanceUID"].tolist():
        raise RuntimeError("study order mismatch")
        
    tr_r=dino_df[blend_labels].rank(method="average",pct=True)
    cr_r=coat_df[blend_labels].rank(method="average",pct=True)
    out=dino_df.copy()
    
    for lbl in blend_labels:
        cw=float(V18_COATNET_WEIGHT[lbl])
        out[lbl]=(1-cw)*tr_r[lbl]+cw*cr_r[lbl]
        
    out[blend_labels]=out[blend_labels].rank(method="average",pct=True)
    vals=out[blend_labels].to_numpy(np.float64)
    
    if not np.isfinite(vals).all() or vals.min()<0 or vals.max()>1:
        raise RuntimeError("invalid blended values")
        
    out.to_csv(_primary,index=False)
    log(f"V18 blend complete → submission.csv {out.shape}")
    
    for _tmp in [WORKING_DIR/"submission_coatnet.csv",
                 WORKING_DIR/"submission_legacy_fold_blend.csv",
                 WORKING_DIR/"legacy_fold_diagnostics.csv"]:
        try:
            if _tmp.is_file(): _tmp.unlink()
        except OSError: pass
except Exception as _blend_err:
    log(f"blend failed: {type(_blend_err).__name__}: {_blend_err}")
    traceback.print_exc()
    if _backup.is_file():
        _shutil.copy2(_backup, _primary)
        log("restored backup submission.csv")
finally:
    try:
        if _backup.is_file(): _backup.unlink()
    except OSError: pass

if not _primary.is_file():
    raise RuntimeError("submission.csv missing after blend")
log("Final submission.csv verified.")

# ── Pseudo-label generation from the blended submission ──────────────────────
PSEUDO_MARGIN       = 0.40
PSEUDO_MIN_FINDS    = 1
PSEUDO_MIN_CONF     = 0.90
GOLD_STUDIES_N      = 58

def generate_pseudo_labels(submission_path, train_ids=None, gold_ids=None,
                           margin=PSEUDO_MARGIN, min_conf=PSEUDO_MIN_CONF, min_findings=PSEUDO_MIN_FINDS):
    """High-confidence pseudo-labels. Exclude gold studies, filter by confidence ≥ min_conf."""
    sub = pd.read_csv(submission_path, dtype={"StudyInstanceUID": str})
    if train_ids is not None:
        sub = sub[~sub["StudyInstanceUID"].isin(train_ids)]
    if gold_ids is not None:
        sub = sub[~sub["StudyInstanceUID"].isin(gold_ids)]
    scores = sub[TARGETS].values
    conf = np.abs(scores - 0.5)
    hc_mask = (conf >= min_conf).sum(axis=1) >= min_findings
    pseudo_df = sub[hc_mask].copy().reset_index(drop=True)
    log(f"Pseudo-labels: {len(pseudo_df)} / {len(sub)} studies (conf ≥ {min_conf}, findings ≥ {min_findings})")
    return pseudo_df

print("Pseudo-label helper defined.")

# ── Fine-Tuning / Training Loop ───────────────────────────────────────────────
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score

class KneeStudyDataset(Dataset):
    def __init__(self, study_df, ser_records, tsdir, res=COAT_IMG, training=True):
        self.study_ids   = study_df["StudyInstanceUID"].astype(str).tolist()
        self.labels      = study_df[TARGETS].values.astype(np.float32)
        self.ser_records = ser_records
        self.tsdir       = tsdir
        self.res         = res
        self.training    = training
        self.reader      = _make_coat_reader()

    def __len__(self):
        return len(self.study_ids)

    def __getitem__(self, idx):
        sid = self.study_ids[idx]
        try:
            vol, mask = build_coat_study(sid, self.ser_records, self.tsdir, self.reader)
            xw        = coat_eval_windows(vol, mask, k=COAT_K_EVAL, res=self.res)
        except Exception:
            xw = torch.zeros(COAT_K_EVAL, 3, self.res, self.res)
        return xw, torch.tensor(self.labels[idx])

class KneeStudyDatasetLowMem(Dataset):
    def __init__(self, study_df, ser_records, tsdir, res=COAT_IMG, k_eval=8, training=True):
        self.study_ids   = study_df["StudyInstanceUID"].astype(str).tolist()
        self.labels      = study_df[TARGETS].values.astype(np.float32)
        self.ser_records = ser_records
        self.tsdir       = tsdir
        self.res         = res
        self.k_eval      = k_eval
        self.training    = training
        self.reader      = _make_coat_reader()

    def __len__(self):
        return len(self.study_ids)

    def __getitem__(self, idx):
        sid = self.study_ids[idx]
        try:
            vol, mask = build_coat_study(sid, self.ser_records, self.tsdir, self.reader)
            xw        = coat_eval_windows(vol, mask, k=self.k_eval, res=self.res)
        except Exception:
            xw = torch.zeros(self.k_eval, 3, self.res, self.res)
        return xw, torch.tensor(self.labels[idx])

# ── Loss Functions & CV Utilities ──────────────────────────────────────────
class FocalLoss(nn.Module):
    """Focal loss for class imbalance. ponytail: minimal implementation."""
    def __init__(self, alpha=1.0, gamma=2.0):
        super().__init__()
        self.alpha, self.gamma = alpha, gamma
    
    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p = torch.sigmoid(logits)
        focal = self.alpha * ((1 - p) ** self.gamma) * bce
        return focal.mean()

from sklearn.model_selection import GroupKFold
def make_cv_splits(study_ids, n_splits=5, seed=SEED):
    """GroupKFold by study_id. Returns list of (train_indices, val_indices)."""
    gkf = GroupKFold(n_splits=n_splits)
    groups = study_ids  # group by study
    splits = []
    for train_idx, val_idx in gkf.split(np.arange(len(study_ids)), groups=groups):
        splits.append((train_idx, val_idx))
    return splits

def train_one_epoch(model, loader, optimizer, scaler, epoch):
    model.train()
    total_loss = 0.0
    criterion  = nn.BCEWithLogitsLoss()
    for i, (xw, labels) in enumerate(loader):
        xw, labels = xw.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with torch.autocast("cuda" if DEVICE.type=="cuda" else "cpu"):
            logits = model(xw)
            loss   = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        total_loss += loss.item()
    return total_loss / max(len(loader), 1)

@torch.inference_mode()
def validate(model, loader):
    """Validate with macro-AUC + per-class AUC tracking."""
    model.eval()
    all_logits, all_labels = [], []
    criterion = nn.BCEWithLogitsLoss()
    for xw, labels in loader:
        xw, labels = xw.to(DEVICE), labels.to(DEVICE)
        logits = model(xw)
        all_logits.append(logits.cpu()); all_labels.append(labels.cpu())
    logits = torch.cat(all_logits); labels = torch.cat(all_labels)
    loss   = criterion(logits, labels).item()
    probs  = torch.sigmoid(logits).numpy()
    y_true = labels.numpy()
    
    try:
        auc = roc_auc_score(y_true, probs, average="macro")
        # ponytail: per-class AUC (minimal, no extra storage)
        per_class_auc = [roc_auc_score(y_true[:, i], probs[:, i]) for i in range(y_true.shape[1])]
    except Exception:
        auc = float("nan")
        per_class_auc = [float("nan")] * len(TARGETS)
    return loss, auc, per_class_auc

# ── Test-Time Augmentation (TTA) ──────────────────────────────────────────
@torch.inference_mode()
def tta_inference(model, xw, n_aug=3):
    """TTA: rotate ±5°, intensity jitter. No flips (anatomy-preserving).
    
    xw: (K, 3, H, W) tensor
    n_aug: number of augmentations to average (includes original)
    Returns: averaged sigmoid probabilities
    """
    probs_list = []
    model.eval()
    
    # Original
    logits = model(xw.to(DEVICE))
    probs_list.append(torch.sigmoid(logits).cpu().numpy())
    
    # Spatial TTA: small rotations ±5°
    if n_aug >= 2:
        for angle_deg in [-5, 5]:
            angle_rad = angle_deg * np.pi / 180
            h, w = xw.shape[-2:]
            cos_a, sin_a = np.cos(angle_rad), np.sin(angle_rad)
            cx, cy = w / 2, h / 2
            M = np.array([[cos_a, -sin_a, cx - cx * cos_a + cy * sin_a],
                         [sin_a, cos_a, cy - cx * sin_a - cy * cos_a + cx]])
            xw_aug = xw.clone()
            for k in range(xw_aug.shape[0]):
                for c in range(3):
                    xw_aug[k, c] = torch.from_numpy(cv2.warpAffine(
                        xw_aug[k, c].numpy(), M, (w, h), borderMode=cv2.BORDER_REPLICATE))
            logits = model(xw_aug.to(DEVICE))
            probs_list.append(torch.sigmoid(logits).cpu().numpy())
    
    # Intensity TTA: brightness ±10%
    if n_aug >= 4:
        xw_bright = (xw.float() * 1.1).clamp(0, 255).to(xw.dtype)
        logits = model(xw_bright.to(DEVICE))
        probs_list.append(torch.sigmoid(logits).cpu().numpy())
    
    # Average and return
    avg_probs = np.stack(probs_list).mean(axis=0)
    return avg_probs

def train_one_epoch_lowmem(model, loader, optimizer, scaler, epoch):
    model.train()
    model.backbone.eval()
    total_loss = 0.0
    criterion  = nn.BCEWithLogitsLoss()
    for i, (xw, labels) in enumerate(loader):
        xw, labels = xw.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with torch.autocast("cuda" if DEVICE.type=="cuda" else "cpu"):
            logits = model(xw)
            loss   = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        total_loss += loss.item()
    return total_loss / max(len(loader), 1)

def run_training(train_df, val_df, ser_records, tsdir,
                 n_epochs=30, lr=1e-4, batch_size=4,
                 weight_decay=1e-2, patience=5,
                 save_path=None):
    
    save_path = save_path or str(WORKING_DIR/"best_raptor.pt")
    arm       = COAT_ARMS[0]
    wp        = find_weight_file(arm["file"])
    model, _  = load_coat_model(wp, arm["arch"], arm["res"], DEVICE)
    model.train()
    
    train_ds = KneeStudyDataset(train_df, ser_records, tsdir, training=True)
    val_ds   = KneeStudyDataset(val_df,   ser_records, tsdir, training=False)
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=0, pin_memory=True)
    val_dl   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    scaler    = torch.amp.GradScaler()
    best_auc, no_improve = 0.0, 0
    history = {"train_loss": [], "val_loss": [], "val_auc": []}
    
    for epoch in range(n_epochs):
        tl       = train_one_epoch(model, train_dl, optimizer, scaler, epoch)
        vl, vauc, _ = validate(model, val_dl)  # ponytail: per-class AUC in 3rd return, unused here
        history["train_loss"].append(tl)
        history["val_loss"].append(vl)
        history["val_auc"].append(vauc)
        flag = ""
        if vauc > best_auc:
            best_auc, no_improve = vauc, 0
            torch.save({"model": model.state_dict(), "arch": arm["arch"], "res": arm["res"]},
                       save_path)
            flag = " ✓ BEST"
        else:
            no_improve += 1
        log(f"Epoch {epoch+1:3d}/{n_epochs} | train {tl:.4f} | val {vl:.4f} | AUC {vauc:.4f}{flag}")
        if no_improve >= patience:
            log(f"Early stopping at epoch {epoch+1}")
            break
        scheduler.step()
        gc.collect()
        if DEVICE.type=="cuda": torch.cuda.empty_cache()
    log(f"Training complete. Best AUC: {best_auc:.4f} → {save_path}")
    return model, history

def run_training_optimized(train_df, val_df, ser_records, tsdir,
                          n_epochs=5, lr=1e-4, batch_size=1, k_eval=8,
                          save_path=None):
    """
    Memory-optimized training loop with frozen backbone.
    
    Fixes applied:
    1. DataLoader multiprocessing (num_workers based on CPU count) for parallel DICOM reading
    2. Explicit model.backbone.eval() after model.train() for correct frozen BN/Dropout behavior
    3. Removed manual gc.collect() and torch.cuda.empty_cache() to prevent allocator interference
    """
    
    save_path = save_path or str(WORKING_DIR / "best_raptor.pt")
    arm       = COAT_ARMS[0]
    wp        = find_weight_file(arm["file"])
    model, _  = load_coat_model(wp, arm["arch"], arm["res"], DEVICE)
    
    # Freeze backbone parameters
    for param in model.backbone.parameters():
        param.requires_grad = False
    
    # Set model to training mode, but backbone to eval for frozen BN/Dropout
    model.train()
    model.backbone.eval()
    
    train_ds = KneeStudyDatasetLowMem(train_df, ser_records, tsdir, res=arm["res"], k_eval=k_eval, training=True)
    val_ds   = KneeStudyDatasetLowMem(val_df,   ser_records, tsdir, res=arm["res"], k_eval=k_eval, training=False)
    
    # Multiprocessing DataLoaders for parallel DICOM reading
    num_workers = max(2, os.cpu_count() or 2)
    
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  
                         num_workers=num_workers, pin_memory=True)
    val_dl   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, 
                         num_workers=num_workers, pin_memory=True)
    
    # Optimize only trainable head parameters
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=1e-2)
    scaler    = torch.amp.GradScaler()
    
    best_auc = 0.0
    history = {"train_loss": [], "val_loss": [], "val_auc": []}
    
    for epoch in range(n_epochs):
        tl = train_one_epoch_lowmem(model, train_dl, optimizer, scaler, epoch)
        vl, vauc, _ = validate(model, val_dl)  # ponytail: per-class AUC in 3rd return
        
        history["train_loss"].append(tl)
        history["val_loss"].append(vl)
        history["val_auc"].append(vauc)
        
        flag = ""
        if vauc > best_auc:
            best_auc = vauc
            torch.save(
                {"model": model.state_dict(), "arch": arm["arch"], "res": arm["res"]},
                save_path
            )
            flag = " ✓ BEST"
        
        log(f"Epoch {epoch+1:2d}/{n_epochs} | train {tl:.4f} | val {vl:.4f} | AUC {vauc:.4f}{flag}")
    
    log(f"Training complete. Best AUC: {best_auc:.4f} → {save_path}")
    return model, history

print("Training helpers defined.")

# ── Submission Verification ───────────────────────────────────────────────────
sub = pd.read_csv(WORKING_DIR/"submission.csv", dtype={"StudyInstanceUID": str})
assert list(sub.columns[:1]) == ["StudyInstanceUID"]
assert set(TARGETS).issubset(sub.columns), "missing target columns"
assert np.isfinite(sub[TARGETS].values).all(), "non-finite values"
assert (sub[TARGETS].values >= 0).all() and (sub[TARGETS].values <= 1).all(), "out of [0,1]"

print(f"submission.csv OK: {sub.shape[0]} studies × {sub.shape[1]} columns")
print(sub.head().to_string(index=False))

✅ Ready to load: /kaggle/input/datasets/dreaddevelopment/raptor-knee-maxspan/raptor_ft_coatnet_v5_full_swa.pt
device cuda:0 | gpus 2 | torch 2.10.0+cu128
[    0.1s] test root: /kaggle/input/competitions/rsna-knee-abnormality-detection | series dir: /kaggle/input/competitions/rsna-knee-abnormality-detection/test_series
[    0.1s] test studies 3 | test series 15
[   44.8s] CoAtNet done → /kaggle/working/submission_coatnet.csv | 3 rows
[   44.8s] DINOv2 package: 20 member(s)
[   44.9s] test header pass: 15 series
[   44.9s] test lat: 1 from tag, 2 geometry, 0 unresolved, 0 disagree
[   44.9s] test: cache (3, 6, 3, 336, 336) = 0.0 GB
[   44.9s] test: ordering 12 slot-series
[   46.7s] test: decoding 12 slot-series
[   46.9s] test: 12/12 slots filled


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

[   62.8s] DINOv2 backbone 12 blocks, last 6 unfrozen, dim 384
[   63.7s] 44128e3ff3: fingerprint OK Δ=4.3e-06
[   64.2s]   banked 44128e3ff3 (1 total)


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

[   65.1s] DINOv2 backbone 12 blocks, last 6 unfrozen, dim 384
[   65.6s] 1e553ba481: fingerprint OK Δ=1.7e-06
[   66.1s]   banked 1e553ba481 (2 total)


...

[   91.2s] DINOv2 backbone 12 blocks, last 6 unfrozen, dim 384
[   91.6s] ea3fbf6edf: fingerprint OK Δ=3.8e-06
[   92.0s]   banked ea3fbf6edf (18 total)


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

[   92.8s] DINOv2 backbone 12 blocks, last 6 unfrozen, dim 384
[   93.1s] 84079fe8cb: fingerprint OK Δ=1.2e-06
[   93.5s]   banked 84079fe8cb (19 total)


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

[   94.3s] DINOv2 backbone 12 blocks, last 6 unfrozen, dim 384
[   94.7s] 3d1f4c48fc: fingerprint OK Δ=1.3e-06
[   95.1s]   banked 3d1f4c48fc (20 total)
[   95.4s] DINOv2 done → submission.csv | (3, 13)
[   95.4s] V18 blend complete → submission.csv (3, 13)
[   95.4s] Final submission.csv verified.
Pseudo-label helper defined.
Training helpers defined.
submission.csv OK: 3 studies × 13 columns
                                                StudyInstanceUID      ACL      MCL  Medial Meniscus  Lateral Meniscus  Medial OA  Lateral OA    PF OA  Effusion  Synovitis  Baker's  Contusion  Fracture
1.2.826.0.1.3680043.8.498.10047035057544427318018579121635276191 0.333333 0.333333         0.666667          0.333333   0.666667    0.333333 0.333333       0.5   0.333333 1.000000   0.333333  0.333333
1.2.826.0.1.3680043.8.498.10062861783145312629332250977456991776 0.833333 1.000000         0.666667          1.000000   1.000000    1.000000 0.833333       1.0   1.000000 0.666667   1.000000  0.833333
